In [4]:
import sys
sys.path.append('/home/jorge.gacitua/salidas/Sensitivity_Experiments/src/sensiwrf')
import argparse
import csv
import os
import shutil
import tempfile

import numpy as np
import matplotlib.pyplot as plt

from wrf_module import read_input_sounding, modify_input_sounding



def map_row_to_conf(row):
    """Mirror the 5-param mapping used during batch generation."""
    p1, p2, p3, p4, p5 = map(float, row)

    conf_dict = {}
    # Wind (Curved)
    conf_dict['modify_wind_profile'] = True
    conf_dict['remove_mean_wind']    = True
    conf_dict['shear_type']          = 'Curved'
    conf_dict['total_shear_depth']   = 4000.0 + (p1 * (8000.0 - 4000.0))
    conf_dict['int_total_shear']     = 1.0e-5 + (p2 * (30.0 - 0.0))
    conf_dict['curved_shear_per']    = p3

    # Linear components (required keys)
    conf_dict['shear_depth_u']    = 0.0
    conf_dict['shear_strength_u'] = 5.0e-3
    conf_dict['shear_depth_v']    = 8000.0
    conf_dict['shear_strength_v'] = 0.0

    # LLJ off by default
    conf_dict['llj_amp']   = 0.0
    conf_dict['llj_h']     = 1500.0
    conf_dict['llj_width'] = 500.0
    conf_dict['llj_dir']   = 360.0

    # Surface wind offsets
    conf_dict['surf_u'] = 0.0
    conf_dict['surf_v'] = 0.0

    # Stability
    conf_dict['modify_stability'] = True
    conf_dict['stability_factor'] = -0.5 + (p4 * (3.0 - -0.5))
    conf_dict['stability_factor_height'] = 5000.0

    # Moisture
    conf_dict['modify_moisture_profile'] = True
    conf_dict['dry_run'] = False
    conf_dict['low_level_moisture_height'] = 2000.0
    conf_dict['low_level_moisture_mult_factor'] = -15.0 + (p5 * (30.0 - -15.0))
    conf_dict['mid_level_moisture_height'] = 2000.0
    conf_dict['mid_level_moisture_mult_factor'] = 0.0

    return conf_dict


def qv_to_dewpoint(p_hpa, T_K, qv_gkg):
    """Convert (p [hPa], T [K], qv [g/kg]) -> dewpoint [K], assuming qv is mixing ratio (kg/kg)."""
    epsilon = 0.622  # R_d/R_v
    r = np.asarray(qv_gkg, dtype=float) / 1000.0  # kg/kg
    p_pa = np.asarray(p_hpa, dtype=float) * 100.0
    e_pa = (r / (epsilon + r)) * p_pa  # vapor pressure
    e_hpa = e_pa / 100.0
    ln_ratio = np.log(np.maximum(e_hpa, 1e-6) / 6.112)
    Td_C = (243.5 * ln_ratio) / (17.67 - ln_ratio)
    Td = Td_C + 273.15
    return np.asarray(Td, dtype=float)


def _interp_to(z_src, x_src, z_target):
    """1D linear interpolation of x(z) from z_src to z_target (both in meters)."""
    return np.interp(z_target, z_src.astype(float), x_src.astype(float))


def load_rows_from_csv(csv_path):
    rows = []
    with open(csv_path, newline='') as f:
        reader = csv.reader(f)
        first = next(reader)
        try:
            _ = [float(x) for x in first]
            rows.append(first)
        except Exception:
            pass
        for r in reader:
            rows.append(r)
    return rows


def maybe_subset_indices(rows, indices_txt):
    if indices_txt is None:
        return list(range(len(rows)))
    idx = []
    with open(indices_txt) as f:
        for line in f:
            s = line.strip()
            if s:
                idx.append(int(s))
    idx = [i for i in idx if 0 <= i < len(rows)]
    return idx

In [6]:
# same CSV you use with run_multiple.py

csv_file = "/home/jorge.gacitua/salidas/Sensitivity_Experiments/experiments/2025-09_supercell_v1/design.csv"
modelpath = "/home/jorge.gacitua/salidas/em_quarter_ss"
base_input = "input_sounding"
max_rows = 800 

base_path = os.path.join(modelpath, base_input)
if not os.path.exists(base_path):
    raise FileNotFoundError(f'Base sounding not found: {base_path}')

# Read base sounding (thick line)
base = read_input_sounding(base_path)
zb = np.asarray(base['height'], dtype=float)  # m
Tb = np.asarray(base['t'], dtype=float)       # K
qvb = np.asarray(base['qv'], dtype=float)     # g/kg
pb = np.asarray(base['p'], dtype=float)       # hPa
ub = np.asarray(base['u'], dtype=float)       # m/s
vb = np.asarray(base['v'], dtype=float)       # m/s
Tdb = qv_to_dewpoint(pb, Tb, qvb)

# Prepare modified ensemble profiles
rows = load_rows_from_csv(csv)
sel_idx = maybe_subset_indices(rows)
if max_rows is not None:
    sel_idx = sel_idx[:max_rows]

# Interpolate all modified profiles onto the base z grid (zb)
T_list, Td_list, qv_list, wspd_list = [], [], [], []

tmpdir = tempfile.mkdtemp(prefix='plot_soundings_env_')
try:
    for ridx in sel_idx:
        row = rows[ridx]
        conf = map_row_to_conf(row)

        # Create a temp copy of the base, modify in-place, then read back
        sound_file = os.path.join(tmpdir, f'input_sounding_{ridx:05d}')
        shutil.copy2(base_path, sound_file)
        modify_input_sounding(sound_file, conf)
        mod = read_input_sounding(sound_file)

        z = np.asarray(mod['height'], dtype=float)
        T = np.asarray(mod['t'], dtype=float)
        qv = np.asarray(mod['qv'], dtype=float)
        p  = np.asarray(mod['p'], dtype=float)
        u  = np.asarray(mod['u'], dtype=float)
        v  = np.asarray(mod['v'], dtype=float)
        Td = qv_to_dewpoint(p, T, qv)
        wspd = np.sqrt(u**2 + v**2)

        # Interpolate to base grid
        T_list.append(_interp_to(z, T, zb))
        Td_list.append(_interp_to(z, Td, zb))
        qv_list.append(_interp_to(z, qv, zb))
        wspd_list.append(_interp_to(z, wspd, zb))
finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

if len(T_list) == 0:
    raise RuntimeError("No rows processed. Check --csv, --indices_txt, or --max_rows.")

T_arr    = np.vstack(T_list)   # [N, nz]
Td_arr   = np.vstack(Td_list)
qv_arr   = np.vstack(qv_list)
wspd_arr = np.vstack(wspd_list)

# Envelopes
T_min, T_max         = np.min(T_arr, axis=0), np.max(T_arr, axis=0)
Td_min, Td_max       = np.min(Td_arr, axis=0), np.max(Td_arr, axis=0)
qv_min, qv_max       = np.min(qv_arr, axis=0), np.max(qv_arr, axis=0)
wspd_min, wspd_max   = np.min(wspd_arr, axis=0), np.max(wspd_arr, axis=0)

# Units
z_km = zb / 1000.0
Tb_C  = Tb - 273.15
Tdb_C = Tdb - 273.15
T_min_C, T_max_C       = T_min - 273.15, T_max - 273.15
Td_min_C, Td_max_C     = Td_min - 273.15, Td_max - 273.15

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
axT, axW, axQ = axes

# 1) T/Td
axT.fill_betweenx(z_km, T_min_C, T_max_C, alpha=0.25, label='T range (mod)')
axT.fill_betweenx(z_km, Td_min_C, Td_max_C, alpha=0.15, label='Td range (mod)')
axT.plot(Tb_C, z_km, lw=2.5, label='Base T')
axT.plot(Tdb_C, z_km, lw=2.5, ls='--', label='Base Td')
axT.set_xlabel('Temperature [°C]')
axT.set_ylabel('Height [km]')
axT.grid(True, alpha=0.3)
axT.legend(loc='best', fontsize=9)

# 2) Wind speed
wspd_b = np.sqrt(ub**2 + vb**2)
axW.fill_betweenx(z_km, wspd_min, wspd_max, alpha=0.25, label='|V| range (mod)')
axW.plot(wspd_b, z_km, lw=2.5, label='Base |V|')
axW.set_xlabel('Wind speed [m s$^{-1}$]')
axW.grid(True, alpha=0.3)
axW.legend(loc='best', fontsize=9)

# 3) qv
axQ.fill_betweenx(z_km, qv_min, qv_max, alpha=0.25, label='qv range (mod)')
axQ.plot(qvb, z_km, lw=2.5, label='Base qv')
axQ.set_xlabel('qv [g kg$^{-1}$]')
axQ.grid(True, alpha=0.3)
axQ.legend(loc='best', fontsize=9)

fig.suptitle('Base sounding (thick) vs range of modified soundings (fill)')
fig.tight_layout(rect=[0, 0.02, 1, 0.96])

TypeError: expected str, bytes or os.PathLike object, not module